In [1]:
%%tsql -artifact ecommerce_lakehouse -type Lakehouse

IF SCHEMA_ID('fda') IS NULL EXEC('CREATE SCHEMA fda')




In [2]:
%%tsql -artifact ecommerce_lakehouse -type Lakehouse

-- Drop existing views
DROP VIEW IF EXISTS [fda].[Sales];
DROP VIEW IF EXISTS [fda].[Date];
DROP VIEW IF EXISTS [fda].[Customers];
DROP VIEW IF EXISTS [fda].[Products];
DROP VIEW IF EXISTS [fda].[Sellers];
DROP VIEW IF EXISTS [fda].[Geography];

In [3]:
%%tsql -artifact ecommerce_lakehouse -type Lakehouse

-- =====================================================
-- DIMENSION: Date
-- Simple date table for role-playing dimensions
-- =====================================================
CREATE VIEW [fda].[Date]
AS
SELECT 
    date_key AS [Date Key],
    date AS [Date],
    year AS [Year],
    quarter AS [Quarter],
    month_number AS [Month Number],
    month_name AS [Month Name],
    year_month AS [Year Month],
    day_of_month AS [Day of Month],
    day_of_week AS [Day of Week],
    day_name AS [Day Name],
    day_of_year AS [Day of Year],
    is_weekend AS [Is Weekend]
FROM [ecommerce_lakehouse].[dbo].[dim_date];

In [4]:
%%tsql -artifact ecommerce_lakehouse -type Lakehouse

-- =====================================================
-- DIMENSION: Customers
-- =====================================================
CREATE VIEW [fda].[Customers]
AS
WITH ranked AS (
    SELECT 
        customer_unique_id,
        customer_zip_code_prefix,
        customer_city,
        customer_state,
        ROW_NUMBER() OVER (PARTITION BY customer_unique_id ORDER BY customer_id) AS rn
    FROM [ecommerce_lakehouse].[dbo].[customers]
)
SELECT 
    customer_unique_id AS [Customer ID],
    customer_zip_code_prefix AS [Customer Zip Code],
    UPPER(customer_city) AS [Customer City],
    UPPER(customer_state) AS [Customer State]
FROM ranked
WHERE rn = 1;

In [5]:
%%tsql -artifact ecommerce_lakehouse -type Lakehouse
-- =====================================================
-- DIMENSION: Products
-- =====================================================
CREATE VIEW [fda].[Products]
AS
SELECT 
    p.product_id AS [Product ID],
    p.product_category_name AS [Product Category],
    COALESCE(t.product_category_name_english, p.product_category_name) AS [Product Category English],
    p.product_name_lenght AS [Product Name Length],
    p.product_description_lenght AS [Product Description Length],
    p.product_photos_qty AS [Product Photos Count],
    p.product_weight_g AS [Product Weight (g)],
    p.product_length_cm AS [Product Length (cm)],
    p.product_height_cm AS [Product Height (cm)],
    p.product_width_cm AS [Product Width (cm)]
FROM [ecommerce_lakehouse].[dbo].[products] p
LEFT JOIN [ecommerce_lakehouse].[dbo].[product_category_translation] t 
    ON p.product_category_name = t.product_category_name;

In [6]:
%%tsql -artifact ecommerce_lakehouse -type Lakehouse
-- =====================================================
-- DIMENSION: Sellers
-- =====================================================
CREATE VIEW [fda].[Sellers]
AS
SELECT 
    seller_id AS [Seller ID],
    seller_zip_code_prefix AS [Seller Zip Code],
    UPPER(seller_city) AS [Seller City],
    UPPER(seller_state) AS [Seller State]
FROM [ecommerce_lakehouse].[dbo].[sellers];

In [7]:
%%tsql -artifact ecommerce_lakehouse -type Lakehouse
-- =====================================================
-- DIMENSION: Geography
-- =====================================================
CREATE VIEW [fda].[Geography]
AS
SELECT 
    geolocation_zip_code_prefix AS [Zip Code],
    UPPER(MAX(geolocation_city)) AS [City],
    UPPER(MAX(geolocation_state)) AS [State],
    AVG(geolocation_lat) AS [Latitude],
    AVG(geolocation_lng) AS [Longitude]
FROM [ecommerce_lakehouse].[dbo].[geolocation]
GROUP BY geolocation_zip_code_prefix;

In [8]:
%%tsql -artifact ecommerce_lakehouse -type Lakehouse

-- =====================================================
-- FACT: Sales
-- Denormalized with Payment Type + Computed Columns
-- =====================================================
CREATE VIEW [fda].[Sales]
AS
WITH payments_agg AS (
    SELECT 
        order_id,
        SUM(payment_value) AS total_payment_value,
        MAX(payment_installments) AS payment_installments,
        MIN(payment_type) AS payment_type
    FROM [ecommerce_lakehouse].[dbo].[order_payments]
    GROUP BY order_id
)
SELECT 
    -- IDs
    oi.order_id AS [Order ID],
    oi.order_item_id AS [Order Item ID],
    oi.product_id AS [Product ID],
    oi.seller_id AS [Seller ID],
    c.customer_unique_id AS [Customer ID],
    c.customer_zip_code_prefix AS [Customer Zip Code],
    
    -- Payment (denormalized)
    p.payment_type AS [Payment Type],
    p.payment_installments AS [Payment Installments],
    
    -- Date Keys (for joining to Date dimension)
    CAST(FORMAT(o.order_purchase_timestamp, 'yyyyMMdd') AS INT) AS [Order Date Key],
    CAST(FORMAT(o.order_approved_at, 'yyyyMMdd') AS INT) AS [Approved Date Key],
    CAST(FORMAT(oi.shipping_limit_date, 'yyyyMMdd') AS INT) AS [Shipping Limit Date Key],
    CAST(FORMAT(o.order_delivered_carrier_date, 'yyyyMMdd') AS INT) AS [Carrier Delivery Date Key],
    CAST(FORMAT(o.order_delivered_customer_date, 'yyyyMMdd') AS INT) AS [Customer Delivery Date Key],
    CAST(FORMAT(o.order_estimated_delivery_date, 'yyyyMMdd') AS INT) AS [Estimated Delivery Date Key],
    
    -- Actual Dates (for easy filtering)
    CAST(o.order_purchase_timestamp AS DATE) AS [Order Date],
    CAST(o.order_approved_at AS DATE) AS [Approved Date],
    CAST(o.order_delivered_carrier_date AS DATE) AS [Carrier Delivery Date],
    CAST(o.order_delivered_customer_date AS DATE) AS [Customer Delivery Date],
    CAST(o.order_estimated_delivery_date AS DATE) AS [Estimated Delivery Date],
    
    -- Time Attributes (for easy grouping without Date join)
    YEAR(o.order_purchase_timestamp) AS [Order Year],
    DATEPART(QUARTER, o.order_purchase_timestamp) AS [Order Quarter],
    MONTH(o.order_purchase_timestamp) AS [Order Month],
    DATENAME(MONTH, o.order_purchase_timestamp) AS [Order Month Name],
    FORMAT(o.order_purchase_timestamp, 'yyyy-MM') AS [Order Year-Month],
    DATENAME(WEEKDAY, o.order_purchase_timestamp) AS [Order Day Name],
    CASE WHEN DATEPART(WEEKDAY, o.order_purchase_timestamp) IN (1, 7) THEN 1 ELSE 0 END AS [Is Weekend Order],
    
    -- Base Measures
    oi.price AS [Price],
    oi.freight_value AS [Freight Value],
    p.total_payment_value AS [Total Payment Value],
    
    -- Computed Measures
    oi.price + oi.freight_value AS [Line Total],
    DATEDIFF(day, o.order_purchase_timestamp, o.order_delivered_customer_date) AS [Delivery Days],
    DATEDIFF(day, o.order_estimated_delivery_date, o.order_delivered_customer_date) AS [Delivery Variance Days],
    DATEDIFF(hour, o.order_purchase_timestamp, o.order_approved_at) AS [Approval Hours],
    
    -- Computed Flags
    CASE WHEN o.order_status = 'delivered' THEN 1 ELSE 0 END AS [Is Delivered],
    CASE 
        WHEN o.order_delivered_customer_date IS NULL THEN NULL
        WHEN o.order_delivered_customer_date > o.order_estimated_delivery_date THEN 1 
        ELSE 0 
    END AS [Is Late],
    CASE 
        WHEN o.order_delivered_customer_date IS NULL THEN 'Pending'
        WHEN o.order_delivered_customer_date <= o.order_estimated_delivery_date THEN 'On Time'
        ELSE 'Late'
    END AS [Delivery Status],
    
    -- Degenerate Dimension
    o.order_status AS [Order Status]
    
FROM [ecommerce_lakehouse].[dbo].[order_items] oi
INNER JOIN [ecommerce_lakehouse].[dbo].[orders] o 
    ON oi.order_id = o.order_id
LEFT JOIN [ecommerce_lakehouse].[dbo].[customers] c 
    ON o.customer_id = c.customer_id
LEFT JOIN payments_agg p 
    ON oi.order_id = p.order_id;

In [9]:
%%tsql -artifact ecommerce_lakehouse -type Lakehouse
-- =====================================================
-- Validation: Row Counts
-- =====================================================
SELECT 'Sales' AS [View], COUNT(*) AS [Rows] FROM [fda].[Sales]
UNION ALL
SELECT 'Date', COUNT(*) FROM [fda].[Date]
UNION ALL
SELECT 'Customers', COUNT(*) FROM [fda].[Customers]
UNION ALL
SELECT 'Products', COUNT(*) FROM [fda].[Products]
UNION ALL
SELECT 'Sellers', COUNT(*) FROM [fda].[Sellers]
UNION ALL
SELECT 'Geography', COUNT(*) FROM [fda].[Geography];